# Stage 2.4 — BatchNorm + manual backprop

**Hardest stage. The point.**

Stage 2.3 trained an MLP with autograd. Now you build a 6-layer net **and manually derive the backward pass for every layer**, then verify against autograd. This is the lecture where Karpathy says *"by the end of this video you will be a backprop ninja."*

It's also where BatchNorm enters. BatchNorm is the architectural change that makes deep nets actually trainable — the saturation of tanh activations is the bug; BatchNorm is the fix. Stage 2.4 has you derive its backward pass by hand.

## What you'll ship

1. A **6-layer MLP** with embedding → 5 hidden `linear → batchnorm → tanh` blocks → logits
2. **Forward pass** built layer-by-layer
3. **Manual backward** — for every layer, derive `dW`, `db`, `dx`, and (for BatchNorm) `d(gain)`, `d(bias)` from `dout` by hand
4. **`cmp(name, dt, t)` helper** that checks your manual gradient against PyTorch's `.grad`
5. All 6 layers' backward passes match autograd to atol=1e-9
6. Train with the manual backward, hit dev loss ≈ 2.05

## Two lectures, in order

1. [makemore Part 3: Activations & Gradients, BatchNorm](https://www.youtube.com/watch?v=P6sfmUTpUmc) — context for *why* BatchNorm exists. Watch this first.
2. [makemore Part 4: Becoming a Backprop Ninja](https://www.youtube.com/watch?v=q8SA3rM6ckI) — the manual-backward drill. This is the stage.

## Why this stage is the most important

Every other stage in this curriculum teaches you to **use** autograd. This one teaches you to **distrust** it. After Stage 2.4 you can read a forward pass and know what backward will look like. Anthropic-style interviews ask exactly this: derive backprop through softmax, or through a specific architecture, on a whiteboard. The drill here is the prep.

## Common traps

1. **Cross-entropy backward** has a beautiful closed form: `dlogits = (softmax(logits) - one_hot(y)) / batch_size`. Don't derive it from logsumexp by hand — derive once, recognize it forever.
2. **BatchNorm backward** is the hardest. The trick is to **NOT** chain through the running statistics; just differentiate the closed-form normalize-and-scale.
3. **`broadcasting` in backward** — when a (1, H) bias is broadcast to (N, H) in forward, its backward sums over the broadcast dim. Easy to miss.
4. **Validate every layer before moving on.** Don't write all 6 backward passes then check at the end — debug one at a time.

## Plan

1. **Forward pass** (use the same architecture as Karpathy's lecture — copy his block structure)
2. For each layer, save the intermediate activations (so backward can access them)
3. **Manual backward** — for each layer write `_backward()` by hand
4. The `cmp` helper:
   ```python
   def cmp(s, dt, t):
       ex = torch.all(dt == t.grad).item()
       app = torch.allclose(dt, t.grad)
       maxdiff = (dt - t.grad).abs().max().item()
       print(f'{s:15s} exact={ex} approx={app} max_diff={maxdiff:.2e}')
   ```
5. Iterate until every `cmp` prints `exact=True`
6. Then train with your manual backward, hit dev loss ~2.05

**Pacing:** this is a 2-day stage. Day one: derive everything, get `cmp` green on a 1-step run. Day two: full training.

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path

# Same vocab setup as previous stages
words = Path('data/names.txt').read_text().splitlines()
chars = sorted(set(''.join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)

# TODO: build_dataset (reuse from Stage 2.3), init 6-layer MLP, run forward,
# derive each layer's backward by hand, verify with cmp() helper.

## End-of-stage check

- [ ] Every `cmp(...)` print shows `exact=True`
- [ ] You can derive `dlogits = (softmax(logits) - one_hot(y)) / batch_size` on a piece of paper without notes
- [ ] You understand why BatchNorm's running statistics don't need a backward (they're not parameters; the gain/bias are)
- [ ] Dev loss ≈ 2.05 with the manually-backproped trainer

One line to `NOTES.md`: which layer's backward was the hardest to derive?

Next: Stage 2.5 — hierarchical context (WaveNet). This is where context length grows from 3 to 8+ characters without making the input layer enormous.